In [198]:
import pandas as pd
from pathlib import Path

In [199]:
# 1. Get the directory THIS script is saved
data_dir = Path.cwd().parent / 'Data'

# 2. Load the files
dataframes= {}
try:
    for file_path in data_dir.glob('*.csv'):
        # Use the file name without the extension (.csv)
        file_name = file_path.stem
        dataframes[file_name] = pd.read_csv(file_path, encoding='latin1')
        print(f'Successfully loaded {file_name} to dataframes')
    
        # Access the data easily
        # roster_df = dataframes['roster']
except Exception as e:
    print(f'An error has occurred: {e}')

Successfully loaded contacts to dataframes
Successfully loaded productivity to dataframes
Successfully loaded roster to dataframes


In [200]:
# Create a copy of dataframes for data cleaning
# Checking all data at once
staging_df = dataframes.copy()
for name, df in staging_df.items():
    print(f'--- {name} ---')
    print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
    print(df.columns.tolist())
    print('\n')

--- contacts ---
Rows: 248397, Columns: 12
['Agent', 'Date', 'LOB', 'Channel', 'ACW', 'Dual/Multiple_Chat_AHT', 'Inbound_Transaction', 'Outbound_Transaction', 'Handle_Time', 'Hold_Time', 'Outbound_Handle_Time', 'Missed_Contacts']


--- productivity ---
Rows: 183860, Columns: 16
['Agent', 'Date', 'Aux_Duration', '1st_BreakDuration', '2nd_Break_Duration', '3rd_Break_Duration', 'Email_Duration', 'Lunch_Duration', 'Meeting_Duration_', 'Technical_Issue_Duration', 'Personal_Duration', 'Task_Duration', 'Training_Duration', 'Available_Duration', 'Busy_Duration', 'Login_Duration']


--- roster ---
Rows: 2544, Columns: 12
['Agent', 'Team Manager', 'Active Date', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Days', 'Tenurity']




In [201]:
# Checking descriptive statics of all at once
for name, df in staging_df.items():
    print(f'--- {name} ---')
    print(df.info())
    print('\n')

--- contacts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248397 entries, 0 to 248396
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Agent                   248397 non-null  object
 1   Date                    248397 non-null  object
 2   LOB                     248397 non-null  object
 3   Channel                 248397 non-null  object
 4   ACW                     248397 non-null  int64 
 5   Dual/Multiple_Chat_AHT  248397 non-null  int64 
 6   Inbound_Transaction     248397 non-null  int64 
 7   Outbound_Transaction    248397 non-null  int64 
 8   Handle_Time             248397 non-null  int64 
 9   Hold_Time               248397 non-null  int64 
 10  Outbound_Handle_Time    248397 non-null  int64 
 11  Missed_Contacts         248397 non-null  int64 
dtypes: int64(8), object(4)
memory usage: 22.7+ MB
None


--- productivity ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1

In [202]:
# --------- DATA CLEANING PHASE --------- 

# Drop Columns with NULL Value only and not needed
staging_df = {key: df.dropna(axis=1, how='all') for key, df in staging_df.items()}
staging_df['roster'] = staging_df['roster'].drop(columns=['Days', 'Tenurity'], errors='ignore')

# Standardize columns headers for easy access
for name, df in staging_df.items():
    staging_df[name].columns = staging_df[name].columns.str.lower().str.strip().str.replace(r'[^\w]+', '_', regex=True).str.strip('_')

In [203]:
# Checking columns header
for name, df in staging_df.items():
    print(f'--- {name} ---')
    print(df.columns.tolist())
    print('\n')

--- contacts ---
['agent', 'date', 'lob', 'channel', 'acw', 'dual_multiple_chat_aht', 'inbound_transaction', 'outbound_transaction', 'handle_time', 'hold_time', 'outbound_handle_time', 'missed_contacts']


--- productivity ---
['agent', 'date', 'aux_duration', '1st_breakduration', '2nd_break_duration', '3rd_break_duration', 'email_duration', 'lunch_duration', 'meeting_duration', 'technical_issue_duration', 'personal_duration', 'task_duration', 'training_duration', 'available_duration', 'busy_duration', 'login_duration']


--- roster ---
['agent', 'team_manager', 'active_date']




In [204]:
# Update `date` columns data type to date
data_cols = ['date', 'active_date']

for name, df in staging_df.items():
    # Convert only if the columns exists
    for col in data_cols:
        if col in df.columns:
            staging_df[name][col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

In [205]:
# Checking Data type of dates
for name, df in staging_df.items():
    print(f'--- {name} ---')
    print(df.info())
    print('\n')

--- contacts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248397 entries, 0 to 248396
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   agent                   248397 non-null  object        
 1   date                    248397 non-null  datetime64[ns]
 2   lob                     248397 non-null  object        
 3   channel                 248397 non-null  object        
 4   acw                     248397 non-null  int64         
 5   dual_multiple_chat_aht  248397 non-null  int64         
 6   inbound_transaction     248397 non-null  int64         
 7   outbound_transaction    248397 non-null  int64         
 8   handle_time             248397 non-null  int64         
 9   hold_time               248397 non-null  int64         
 10  outbound_handle_time    248397 non-null  int64         
 11  missed_contacts         248397 non-null  int64         
dtypes: datetime64

In [206]:
# Check exact count of duplicates across all staging DataFrames
for name, df in staging_df.items():
    dup_count = df.duplicated(keep=False).sum()
    print(f'--- {name} ---')
    print(f'Number of exact duplicates: {dup_count}\n')

--- contacts ---
Number of exact duplicates: 2

--- productivity ---
Number of exact duplicates: 14

--- roster ---
Number of exact duplicates: 0



In [207]:
# --- Deduplication ---

# 1. Handle Exact duplicates
for name, df in staging_df.items():
    staging_df[name] = staging_df[name].drop_duplicates()

for name, df in staging_df.items():
    dup_count = staging_df[name].duplicated(keep=False).sum()
    print(f'--- {name} ---')
    print(f'Number of exact duplicates: {dup_count}')
    print('\n')

--- contacts ---
Number of exact duplicates: 0


--- productivity ---
Number of exact duplicates: 0


--- roster ---
Number of exact duplicates: 0




In [208]:
# 2. Handle logical Duplicates via Aggregation
# For Productivity: One row per Agent per Day
prod_clean = staging_df['productivity'].groupby(['agent', 'date']).sum().reset_index()

# For Contacts: One row per Agent, Date, LOB, and Channel
# We define our 'Business Key' columns first
key_cols = ['agent', 'date', 'lob', 'channel']
contacts_clean = staging_df['contacts'].groupby(key_cols).sum().reset_index()

In [209]:
# Merge file to create a Master file
master_file = pd.merge(contacts_clean, prod_clean, on=['agent', 'date'], how='left')
master_file = pd.merge(master_file, staging_df['roster'], on='agent', how='left')

In [210]:
# Save file in Data for analysis and visualization
# master_file.to_csv('../Data/Master File.csv', index=False)

In [211]:
contacts_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192198 entries, 0 to 192197
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   agent                   192198 non-null  object        
 1   date                    192198 non-null  datetime64[ns]
 2   lob                     192198 non-null  object        
 3   channel                 192198 non-null  object        
 4   acw                     192198 non-null  int64         
 5   dual_multiple_chat_aht  192198 non-null  int64         
 6   inbound_transaction     192198 non-null  int64         
 7   outbound_transaction    192198 non-null  int64         
 8   handle_time             192198 non-null  int64         
 9   hold_time               192198 non-null  int64         
 10  outbound_handle_time    192198 non-null  int64         
 11  missed_contacts         192198 non-null  int64         
dtypes: datetime64[ns](1), int64(8)

In [212]:
prod_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168891 entries, 0 to 168890
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   agent                     168891 non-null  object        
 1   date                      168891 non-null  datetime64[ns]
 2   aux_duration              168891 non-null  int64         
 3   1st_breakduration         168891 non-null  int64         
 4   2nd_break_duration        168891 non-null  int64         
 5   3rd_break_duration        168891 non-null  int64         
 6   email_duration            168891 non-null  int64         
 7   lunch_duration            168891 non-null  int64         
 8   meeting_duration          168891 non-null  int64         
 9   technical_issue_duration  168891 non-null  int64         
 10  personal_duration         168891 non-null  int64         
 11  task_duration             168891 non-null  int64         
 12  tr

In [213]:
staging_df['roster'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2544 entries, 0 to 2543
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   agent         2544 non-null   object        
 1   team_manager  2544 non-null   object        
 2   active_date   2544 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(2)
memory usage: 59.8+ KB


In [184]:
# Check how many unique entries exist for each grouping
print(f"Unique Agent-Dates: {len(staging_df['contacts'].groupby(['agent', 'date']))}")
print(f"Unique Agent-Date-Channels: {len(staging_df['contacts'].groupby(['agent', 'date', 'channel']))}")

Unique Agent-Dates: 66943
Unique Agent-Date-Channels: 78501


In [214]:
# Validation: Sum of metrics should remain identical
original_sum = staging_df['contacts']['handle_time'].sum()
clean_sum = contacts_clean['handle_time'].sum()

print(f"Original Total: {original_sum}")
print(f"Cleaned Total:  {clean_sum}")
print(f"Difference:     {original_sum - clean_sum}")

Original Total: 3169261781
Cleaned Total:  3169261781
Difference:     0
